# Human video vs robot video for a robot world model

Two world models, same code, same compute and the same number of training frames; only the video differs:
- `wm-phone`: human videos filmed with a phone (`batgre/h2r-phone`);
- `wm-robots`: real robot videos (`batgre/h2r-robots-1h30`), the first videos of a seeded shuffle up to the same
  number of frames.

Each gets a controller trained on the same robosuite episodes (Panda, Lift, `batgre/h2r-sim-v3`), then everything is
evaluated: the world models' predictions, the controllers, the planner in closed loop.

Run the cells in order. Runs go to Drive (`MyDrive/h2r/outputs`): after a disconnection, run the cells again,
trainings continue where they stopped and finished evaluations are skipped.

**Setup** (2 min). Clones or updates the repository and installs the environment.

In [ ]:
# colab
!pip install -q uv
%cd /content
![ -d human-to-robot ] || git clone https://github.com/batgr/human-to-robot
%cd /content/human-to-robot
!git pull && uv sync

**Keys and Drive.** Asks for the Hugging Face token and the W&B key, links `outputs/` to Drive.

In [ ]:
# colab
import os
from getpass import getpass

from google.colab import drive

os.environ["HF_TOKEN"] = getpass("HF token: ")
os.environ["WANDB_API_KEY"] = getpass("W&B key: ")
os.environ["MUJOCO_GL"] = "egl"  # offscreen rendering for robosuite
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/h2r/outputs && ln -sfn /content/drive/MyDrive/h2r/outputs outputs

**Settings.** One run = one `RUN` suffix on every output name, so runs never overwrite each other.
- `H`: the world models are trained to unroll `H` steps (`rollout_steps.end`) and the planner looks `H` steps ahead;
  the context stays at 3 frames. Run 1: `RUN = ""`, `H = 5`; run 2: `RUN = "-h2"`, `H = 2`.
- `PHONE_REV`: the version of the phone videos. Run 1 used `668913068cc7` (65 training videos, 2,580 frames); keep it to
  change only the horizon, or `"main"` for every video filmed since.
- `ROBOT_VIDEOS`: as many robot videos as needed to match the phone videos' training frames (102 for run 1).
- Steps: about 60 passes over the world models' clips and 17 over the controllers' transitions; `EPISODES` test
  episodes for the pick.

In [ ]:
import json

from IPython.display import Image, display

RUN, H = "-h2", 2
PHONE_REV, ROBOT_VIDEOS = "668913068cc77e2cb195a8b3a88a4d87fd317fb1", 102  # run 1 data: 2,580 phone and 2,587 robot frames
PHONE = "data.root=data/h2r-phone/ data.hf_repo=batgre/h2r-phone"
ROBOTS = f"data.root=data/h2r-robots-1h30/ data.hf_repo=batgre/h2r-robots-1h30 data.max_videos={ROBOT_VIDEOS}"
WM_STEPS, WM_EVAL, CTRL_STEPS, EPISODES = 1000, 50, 4000, 30

## 1. Data

**Downloads the videos** (a few minutes): the phone frames (without the raw videos) and the robot frames.

In [ ]:
!uv run hf download batgre/h2r-phone --repo-type dataset --revision {PHONE_REV} --local-dir data/h2r-phone --exclude "raw/*"
!uv run hf download batgre/h2r-robots-1h30 --repo-type dataset --local-dir data/h2r-robots-1h30

## 2. World models

**Trains both world models** (about 10-20 min each). Every 50 steps the log prints the validation on the model's own
videos:
- `tf`, `ro`: prediction errors (one step with the true past, several steps on its own predictions); should go down;
- `skill_h1..h5`: 1 - error / error of repeating the last frame, h steps ahead; above 0 = better than no change;
- `controllability`: how much the prediction depends on the latent action (0 = the action is ignored);
- `rank`, `active_dims`: richness of the states and of the latent actions (falling to a few = collapse);
- `straightness`: about -0.3 for an untrained model; higher = smoother latent trajectories.

`best.pt` keeps the step with the best mean skill.

In [ ]:
!uv run h2r train --set name=wm-phone{RUN} {PHONE} rollout_steps.end={H} iterations={WM_STEPS} eval_every={WM_EVAL}
!uv run h2r train --set name=wm-robots{RUN} {ROBOTS} rollout_steps.end={H} iterations={WM_STEPS} eval_every={WM_EVAL}

**Training curves** of both world models: every loss and validation metric against the step (`curves.png`).

In [ ]:
for run in [f"wm-phone{RUN}", f"wm-robots{RUN}"]:
    !grep -E "^step|^done" outputs/{run}/log.txt | tail -2
    display(Image(f"outputs/{run}/curves.png", width=1000))

## 3. Controllers

**Trains one controller per world model** on the same robosuite episodes (about 10 min each, after downloading them).
A controller maps a real robot action and the current frame to the world model's latent action. Log: validation loss
and `val r2` (0 = no better than always predicting the mean latent action, 1 = perfect); `best.pt` keeps the lowest
validation loss.

In [ ]:
!uv run hf download batgre/h2r-sim-v3 --repo-type dataset --local-dir data/h2r-sim
for wm in ["phone", "robots"]:
    !uv run h2r controller train --set name=ctrl-{wm}{RUN} world_model=outputs/wm-{wm}{RUN} iterations={CTRL_STEPS} eval_every=100

**Controller curves**: training loss, validation loss and R² against the step.

In [ ]:
for run in [f"ctrl-phone{RUN}", f"ctrl-robots{RUN}"]:
    !grep -E "^done" outputs/{run}/log.txt
    display(Image(f"outputs/{run}/curves.png", width=700))

## 4. World model predictions

**Every world model on every validation set** (about 10 min): the same validation as in training, on the phone, robot
and simulation videos. The only skill numbers comparable across world models; neither model saw simulation videos,
so the `sim` row compares them on equal terms.

In [ ]:
%%writefile /content/cross.py
import json
import sys

from h2r.models.jepa import load_world_model
from h2r.models.module import SIGReg
from h2r.training.utils import download_data, setup
from h2r.training.world_model import build_loaders, evaluate

DOMAINS = {"phone": {"root": "data/h2r-phone/", "hf_repo": "batgre/h2r-phone", "include": None},
           "robots": {"root": "data/h2r-robots-1h30/", "hf_repo": "batgre/h2r-robots-1h30", "include": None},
           "sim": {"root": "data/h2r-sim/", "hf_repo": "batgre/h2r-sim-v3", "include": ["Panda/Lift/*"]}}
device, autocast = setup({"seed": 0, "precision": "bf16"})
table = {}
RUN = sys.argv[1] if len(sys.argv) > 1 else ""
for wm in [f"wm-phone{RUN}", f"wm-robots{RUN}"]:
    model, cfg = load_world_model(f"outputs/{wm}", device)
    for name, data in DOMAINS.items():
        c = cfg | {"data": cfg["data"] | data | {"max_videos": None}}
        download_data(c["data"])
        m = evaluate(model, SIGReg().to(device), build_loaders(c, device)[1], c, device, autocast)
        print(f"{wm:10} on {name:7} skill h1..h5 " + " ".join(f"{m[f'skill_h{h}']:.3f}" for h in range(1, 6))
              + f" | controllability {m['controllability']:.3f} | rank {m['rank']:.1f} | straightness {m['straightness']:.3f}",
              flush=True)
        table[f"{wm} on {name}"] = m
json.dump(table, open(f"outputs/cross{RUN}.json", "w"), indent=2)

In [ ]:
# h2r lives in the uv environment, not in the notebook's Python
!uv run python /content/cross.py {RUN} 2>&1 | grep -v "Loading weights"

**Diagnostics on 10 simulated test episodes** (about 15-20 min), with each world model and its controller:
- `prediction`: skill(h) with the IDM's latent actions (oracle, the best the world model can do), with the controller on
  the robot's actions, and in the loop on its own predictions as when planning; `imagined.png` shows, for each
  predicted state, the nearest real frame;
- `counterfactual`: 8 action sequences from the same state; does each prediction identify its own outcome (chance
  0.12), with the right direction and spread between outcomes? controller vs oracle;
- `ranking`: does the planner's cost rank sequences like their real outcome (Spearman, regret in cm)?

In [ ]:
for wm in ["phone", "robots"]:  # prediction: h1..h5 whatever H, comparable across runs; the others play H-step sequences
    !uv run h2r eval prediction --set name=diag-{wm}{RUN} controller=outputs/ctrl-{wm}{RUN} episodes=10
    for d in ["counterfactual", "ranking"]:
        !uv run h2r eval {d} --set name=diag-{wm}{RUN} controller=outputs/ctrl-{wm}{RUN} episodes=10 horizon={H} planner.horizon={H}

In [ ]:
for wm in ["phone", "robots"]:
    for figure in ["prediction/skill.png", "prediction/imagined.png", "ranking/ranking.png"]:
        display(Image(f"outputs/diag-{wm}{RUN}/{figure}", width=700))

## 5. Pick success in closed loop

**random** (the floor), **replay** (the demonstration's actions, open loop: checks the action interface) and the
**planner** with each world model, on the same test episodes (about 10 min per policy). A pick succeeds when the cube is
lifted 4 cm while grasped; `reached`: the gripper came within 3 cm of the cube; `grasped`: the cube was held. Wilson 95%
intervals.

In [ ]:
!uv run h2r eval pick --set name=pick-baselines "policies=[random, replay]" episodes={EPISODES}  # once for every run
for wm in ["phone", "robots"]:
    !uv run h2r eval pick --set name=pick-{wm}{RUN} "policies=[planner]" controller=outputs/ctrl-{wm}{RUN} episodes={EPISODES} planner.horizon={H}

**Results** of the three pick runs, and the frames of a success and a failure of each planner.

In [ ]:
for run in ["pick-baselines", f"pick-phone{RUN}", f"pick-robots{RUN}"]:
    for policy, result in json.load(open(f"outputs/{run}/results.json"))["summary"].items():
        print(run, policy, result)
for wm in ["phone", "robots"]:
    for kind in ["success", "failure"]:
        if os.path.exists(f"outputs/pick-{wm}{RUN}/episode_{kind}.png"):
            display(Image(f"outputs/pick-{wm}{RUN}/episode_{kind}.png", width=900))

## 6. Results archive

**Every log, curve, result and figure** in `results.zip`, downloaded to your computer.

In [ ]:
!cd outputs && zip -qr /content/results.zip cross*.json */metrics.jsonl */curves.png */results.json */log.txt */metadata.json */*.png diag-*/*/*
from google.colab import files
files.download("/content/results.zip")

## 7. Publish the trained models

Config, best checkpoint and curves of each run, to `batgre/h2r-lift`.

In [ ]:
for run in [f"wm-phone{RUN}", f"wm-robots{RUN}", f"ctrl-phone{RUN}", f"ctrl-robots{RUN}"]:
    !uv run hf upload batgre/h2r-lift outputs/{run} {run} --include config.yaml --include best.pt --include curves.png